# Demo: Hybrid OCR evaluation (SROIE & FUNSD)

**Purpose:** Run **PaddleOCR** via the repo's **HybridOCR** class for **SROIE** and **FUNSD** inference. Works on **Google Colab** (Linux; avoids Windows PaddleOCR dependency issues) or **local/SageMaker**. Mode is **auto-detected** (Colab vs local); use **GPU** on Colab to minimize runtime.

**Critical design:** 3-tier detection (cache → classical → **PaddleOCR**) keeps cost and latency low while escalating hard cases; **HybridOCR** unifies Tesseract + PaddleOCR + optional vision fallback. This notebook uses **only OCR deps** (no QNLP/lambeq/jax) so it stays fast and conflict-free.

**Flow:** Setup → instantiate **HybridOCR** → load SROIE/FUNSD from HuggingFace → run OCR → report/download.

## 1. Setup: Colab vs local

On **Google Colab** this cell clones the repo and installs **paddlepaddle** and **paddleocr** (Linux wheels; avoids Windows dependency hell). Then installs the rest of the repo deps so **HybridOCR** and **ocr_pipeline** are available. On **local** (e.g. Linux/SageMaker) ensure you are in the repo root and have already installed paddle + deps.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    # OCR-only deps (no requirements-notebooks.txt to avoid lambeq/jax / QNLP stack)
    get_ipython().system("pip install -q paddlepaddle paddleocr opencv-python-headless Pillow datasets")
    ROOT = Path(".").resolve()
    print("Colab: repo + PaddleOCR ready. HybridOCR will use PaddleOCR when needed.")
else:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "ocr_pipeline").exists() else Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print("Local/SageMaker: ensure paddlepaddle and paddleocr are installed.")

## 2. Instantiate HybridOCR (PaddleOCR in pipeline)

**HybridOCR** uses 3-tier detection (cache → classical → **PaddleOCR**) and Tesseract/PaddleOCR recognition. When the detection router escalates to tier 3 or when recognition falls back to PaddleOCR, the **PaddleOCRDetector** / PaddleOCR engine is used. This cell verifies that PaddleOCR is available (native or ONNX/microservice).

In [ ]:
from ocr_pipeline.recognition.hybrid_ocr import HybridOCR
import numpy as np

# 3-tier detection (classical + PaddleOCR fallback); no vision augmentation for speed
ocr_system = HybridOCR(use_detection_router=True, use_vision_augmentation=False)

# Quick check: PaddleOCR detector is loaded by DetectionRouter when available
if hasattr(ocr_system, 'detection_router') and ocr_system.detection_router is not None:
    dr = ocr_system.detection_router
    paddle_ok = getattr(dr, 'paddleocr_available', False)
    print("PaddleOCR available in router:", paddle_ok)
else:
    print("HybridOCR ready (detection router may not have PaddleOCR on first run).")
print("HybridOCR instantiated. Run next cells to run on SROIE/FUNSD.")

## 3. Load SROIE and run HybridOCR inference

Load **SROIE** (ICDAR 2019) from HuggingFace (`jsdnrs/ICDAR2019-SROIE`). For each sample we run **HybridOCR.process_document(image)** and optionally compare extracted text/entities to ground truth (company, address, date, total).

In [ ]:
from datasets import load_dataset

SROIE_SPLIT = "train"
MAX_SROIE = 5  # limit samples for a quick run; increase for full eval

ds_sroie = load_dataset("jsdnrs/ICDAR2019-SROIE", split=SROIE_SPLIT)
sroie_samples = list(ds_sroie.select(range(min(MAX_SROIE, len(ds_sroie)))))
print(f"SROIE: {len(sroie_samples)} samples")

sroie_results = []
for i, row in enumerate(sroie_samples):
    img = row.get("image")
    if img is None:
        continue
    if hasattr(img, "convert"):
        img = img.convert("RGB")
    img_np = np.array(img)
    out = ocr_system.process_document(img_np)
    text = out.get("text", "")
    gt_entities = row.get("entities") or {}
    sroie_results.append({
        "sample_id": row.get("key", i),
        "text": text,
        "ground_truth_entities": gt_entities,
        "metadata": out.get("metadata", {}),
    })
    print(f"Sample {i+1}: {len(text)} chars extracted; detection: {out.get('metadata', {}).get('detection_method', 'N/A')}")

print(f"Done. {len(sroie_results)} SROIE samples processed.")

## 4. Load FUNSD and run HybridOCR inference

Load **FUNSD** (Form Understanding) from HuggingFace (`nielsr/funsd`). Run **HybridOCR** on each form image and show extracted text (and optional NER/token comparison if needed).

In [ ]:
FUNSD_SPLIT = "train"
MAX_FUNSD = 5

ds_funsd = load_dataset("nielsr/funsd", split=FUNSD_SPLIT)
funsd_samples = list(ds_funsd.select(range(min(MAX_FUNSD, len(ds_funsd)))))
print(f"FUNSD: {len(funsd_samples)} samples")

funsd_results = []
for i, row in enumerate(funsd_samples):
    img = row.get("image")
    if img is None:
        continue
    if hasattr(img, "convert"):
        img = img.convert("RGB")
    img_np = np.array(img)
    out = ocr_system.process_document(img_np)
    text = out.get("text", "")
    funsd_results.append({
        "sample_id": row.get("id", i),
        "text": text,
        "words_gt": row.get("words", []),
        "metadata": out.get("metadata", {}),
    })
    print(f"Sample {i+1}: {len(text)} chars; detection: {out.get('metadata', {}).get('detection_method', 'N/A')}")

print(f"Done. {len(funsd_results)} FUNSD samples processed.")

## 5. Run eval_runner.py (monitoring metrics)

Run the unified eval runner with `--category ocr` to evaluate SROIE/FUNSD via `run_model` (HybridOCR), record **layout cache hit rate** and **completeness heuristics** per sample, and write:

- `data/proof/monitoring_metrics/layout_fingerprint_cache/layout_fingerprint_cache_per_sample_ocr.json`
- `data/proof/monitoring_metrics/completeness_heuristics/completeness_heuristics_per_sample_ocr.json`
- `data/proof/monitoring_metrics.json` (aggregate)

Run this cell **before** the download cell so the monitoring_metrics folder is populated (and included in the zip download).

In [ ]:
# Run eval_runner for OCR: populates monitoring_metrics per-sample + monitoring_metrics.json
import subprocess
import sys
os.chdir(ROOT)
rc = subprocess.call([
    sys.executable, "eval_runner.py",
    "--category", "ocr",
    "--max_split", "5",
])
if rc != 0:
    print("eval_runner.py finished with exit code", rc)
else:
    print("eval_runner.py completed. monitoring_metrics/ and monitoring_metrics.json updated.")

## 6. Save results and download (Colab)

Write a short **report** and download it plus **data/proof/** artifacts: **monitoring_metrics/** (per-sample JSONs and monitoring_metrics.json) and **ocr/** (SROIE/FUNSD evaluation proofs). Run the **eval_runner** cell above first so both folders are populated.

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sroie_results = sroie_results if 'sroie_results' in dir() else []
funsd_results = funsd_results if 'funsd_results' in dir() else []

report_lines = [
    "PaddleOCR + HybridOCR (SROIE & FUNSD) – run report",
    "=" * 60,
    f"SROIE samples processed: {len(sroie_results)}",
    f"FUNSD samples processed: {len(funsd_results)}",
    "",
]
for r in (sroie_results or [])[:3]:
    report_lines.append(f"SROIE {r.get('sample_id')}: {len(r.get('text', ''))} chars")
for r in (funsd_results or [])[:3]:
    report_lines.append(f"FUNSD {r.get('sample_id')}: {len(r.get('text', ''))} chars")
report_txt = "\n".join(report_lines)
report_path = MODEL_DIR / "report_paddleocr_sroie_funsd.txt"
report_path.write_text(report_txt, encoding="utf-8")
print(report_txt)
print(f"Report saved to {report_path}")

try:
    from google.colab import files
    import zipfile
    files.download(str(report_path))
    print("Download started: save report_paddleocr_sroie_funsd.txt to models/.")
    PROOF_DIR = ROOT / "data" / "proof"
    MONITORING_DIR = PROOF_DIR / "monitoring_metrics"
    MONITORING_DIR.mkdir(parents=True, exist_ok=True)
    for sub in ("layout_fingerprint_cache", "completeness_heuristics", "adversarial_testing"):
        (MONITORING_DIR / sub).mkdir(parents=True, exist_ok=True)
    zip_path = ROOT / "proof_artifacts.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        if (PROOF_DIR / "monitoring_metrics.json").exists():
            zf.write(PROOF_DIR / "monitoring_metrics.json", "monitoring_metrics.json")
        for part in ("layout_fingerprint_cache", "completeness_heuristics", "adversarial_testing"):
            subdir = MONITORING_DIR / part
            for f in subdir.glob("*_per_sample_*.json"):
                zf.write(f, f"monitoring_metrics/{part}/{f.name}")
        OCR_PROOF = PROOF_DIR / "ocr"
        if OCR_PROOF.exists():
            for f in OCR_PROOF.rglob("*"):
                if f.is_file():
                    arcname = str(f.relative_to(PROOF_DIR)).replace("\\", "/")
                    zf.write(f, arcname)
    files.download(str(zip_path))
    print("Download started: proof_artifacts.zip (extract to data/proof/ for monitoring_metrics/, ocr/, and monitoring_metrics.json).")
except ImportError:
    print("Not on Colab. Report is at", str(report_path))